In [0]:
# country sales rankings plus cusomer count
query = """
WITH country_totals AS (
    SELECT 
        c.country,
        SUM(f.sales_amount) AS total_sales,
        COUNT(DISTINCT f.order_number) AS total_orders,
        COUNT(DISTINCT f.customer_sk) AS unique_customers
    FROM bike_data.gold.fact_sales f
    JOIN bike_data.gold.dim_customer c ON f.customer_sk = c.customer_id
    GROUP BY 1
),
global_total AS (
    SELECT SUM(total_sales) AS grand_total FROM country_totals
)
SELECT 
    t.country,
    t.total_sales,
    t.total_orders,
    t.unique_customers,
    -- Ranking countries by sales volume
    RANK() OVER (ORDER BY t.total_sales DESC) AS country_rank,
    -- Percentage of total global revenue
    ROUND((t.total_sales / g.grand_total) * 100, 2) AS pct_of_global_revenue
FROM country_totals t
CROSS JOIN global_total g
ORDER BY country_rank ASC;
"""
df = spark.sql(query)

In [0]:
df.display()